In [1]:
import sys
from pathlib import Path

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, features, models

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, confusion_matrix
import keras
from keras.callbacks import EarlyStopping
import joblib

I0000 00:00:1795602380.335025   53163 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
def construire_fenetres_brutes(selection, canal="DE", duree=0.17067, recouvrement=0):
    SB, lignes = [], []
    for nom, info in selection.iterrows():
        s = data.charger(nom, canaux=(canal,))
        for i, val in enumerate(features.fenetrer(s[canal], s["fs"], duree=duree, recouvrement=recouvrement)):
            SB.append(val)
            lignes.append({"fichier": nom, "fenetre": i, "canal": canal,
                           "defaut": info["defaut"], "charge_hp": info["charge_hp"],
                           "rpm": s["rpm"], "fs": s["fs"],
                           "defaut_sain": "Sain" if info["defaut"] == "Sain" else "Défaut",
                           "diametre_mils": info["diametre_mils"],
                           "classe": "Sain" if info["defaut"] == "Sain" else info["defaut"] + str(info["diametre_mils"]).zfill(3)})
    return np.array(SB, dtype=np.float32), pd.DataFrame(lignes)

def score_mse(modele, X):
    Xr = modele.predict(X[..., None], verbose=0)[..., 0]
    return ((X - Xr) ** 2).mean(axis=1)

def ajouter_bruit(x, snr_db, seed=42):
    rng = np.random.default_rng(seed)
    p = np.mean(x ** 2)
    return x + np.sqrt(p / 10 ** (snr_db / 10)) * rng.standard_normal(x.shape)

# --- DE + FE ---
selection_DE = data.selectionner(sources=("12kDE", "Normal"), defauts=("IR","B","OR"),
                                 diametres=(7,14,21), positions_or=("6",), charges=(0,1,2,3),
                                 canaux={"12kDE": ("DE",), "Normal": ("DE",)})
selection_FE = data.selectionner(sources=("12kFE", "Normal"), defauts=("IR","B","OR"),
                                 diametres=(7,14,21), positions_or=("3","6","12"), charges=(0,1,2,3),
                                 canaux={"12kFE": ("FE",), "Normal": ("FE",)})
SB_DE, infos_DE = construire_fenetres_brutes(selection_DE, canal="DE")
SB_FE, infos_FE = construire_fenetres_brutes(selection_FE, canal="FE")

SB = np.concatenate([SB_DE, SB_FE], axis=0)
infos = pd.concat([infos_DE, infos_FE], ignore_index=True)

print(SB.shape); print(infos.groupby(["canal", "defaut_sain"]).size())

(6431, 2048)
canal  defaut_sain
DE     Défaut         2125
       Sain            828
FE     Défaut         2650
       Sain            828
dtype: int64


In [3]:
# groups = infos["fichier"] + "_" + infos["canal"]
# gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=40)
# i_train, i_test = next(gss.split(infos, groups=groups))

# sain = (infos["defaut_sain"] == "Sain").to_numpy()

# i_sain_FE = np.where((infos["canal"] == "FE") & sain)[0]
# i_test = np.union1d(i_test, i_sain_FE[: len(i_sain_FE) // 4])   # 25 % des sains FE au test
# i_train = np.setdiff1d(np.arange(len(infos)), i_test)           # split FINAL

# # === Scaler fit sur les sains du train uniquement ===
# scaler = StandardScaler().fit(SB[i_train][sain[i_train]])
# SBNorm = scaler.transform(SB)

# X_train_sain = SBNorm[i_train][sain[i_train]]

In [4]:
from sklearn.model_selection import StratifiedGroupKFold

sain = (infos["defaut_sain"] == "Sain").to_numpy()

# === Split par FICHIER entier (plus de fuite inter-capteurs : DE et FE d'un même fichier restent ensemble),
#     stratifié sur sain/défaut => des fichiers SAINS garantis au test (donc sains DE *et* FE) ===
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=40)   # 4 plis => ~25 % des fichiers au test
i_train, i_test = next(cv.split(infos, infos["defaut_sain"], groups=infos["fichier"]))

# --- Vérifications anti-fuite (le notebook plante si l'une échoue) ---
assert len(np.intersect1d(infos["fichier"].to_numpy()[i_train],
                          infos["fichier"].to_numpy()[i_test])) == 0, "fuite : un fichier est des deux côtés"
assert sain[i_test].any(), "aucun fichier sain au test"
assert ((infos["canal"].to_numpy() == "FE") & sain)[i_test].any(), "aucun sain FE au test"

SBNorm = features.standardiser_par_fenetre(SB) # z-score par fenêtre, sans état

X_train_sain = SBNorm[i_train][sain[i_train]]                # max de données saines pour le fit

In [5]:
pd.DataFrame({
    "train": infos.iloc[i_train[sain[i_train]]][["defaut_sain","canal"]].value_counts(),
    "test":  infos.iloc[i_test][["defaut_sain","canal"]].value_counts(),
}).fillna(0).astype(int)

train  test
defaut_sain canal             
Défaut      DE         0   531
            FE         0   648
Sain        DE       591   237
            FE       591   237

In [6]:
def evaluer_par_capteur(modele, tag):
    score = score_mse(modele, SBNorm)
    seuil = np.percentile(score[i_train][sain[i_train]], 99)
    s_test, y = score[i_test], (infos["defaut_sain"].to_numpy()[i_test] == "Défaut").astype(int)
    c = infos["canal"].to_numpy()[i_test]
    lignes = []
    for canal in ["DE", "FE"]:
        m = c == canal
        TN, FP, FN, TP = confusion_matrix(y[m], s_test[m] > seuil, labels=[0, 1]).ravel()
        lignes.append({"config": tag, "capteur": canal, "seuil_p99": round(float(seuil), 4),
                       "AUC": round(roc_auc_score(y[m], s_test[m]), 3),
                       "FPR": round(FP/(FP+TN), 4), "FNR": round(FN/(FN+TP), 4)})
    return pd.DataFrame(lignes)

# Data augmentation : on duplique 4 fois le train sain, en ajoutant du bruit à différentes puissance, la cible reste "propre"
X_aug, Y_aug = [X_train_sain], [X_train_sain]
for snr in [15, 10, 5, 0]:
    X_aug.append(ajouter_bruit(X_train_sain, snr, seed=snr))
    Y_aug.append(X_train_sain)
X_aug, Y_aug = np.concatenate(X_aug), np.concatenate(Y_aug)


# === Max fit avant d'enregistrer le modèle === #
modele_DAE = models.conv1d_ae(SBNorm.shape[1])
modele_DAE.fit(X_aug[..., None], Y_aug[..., None], epochs=100, batch_size=64,
               validation_split=0.1, verbose=1,
               callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)])    # === min_delta=0.02

W0000 00:00:1795602389.472687   53163 cpu_allocator_impl.cc:174] Allocation of 43573248 exceeds 10% of free system memory.
W0000 00:00:1795602389.648680   53163 cpu_allocator_impl.cc:174] Allocation of 43573248 exceeds 10% of free system memory.


Epoch 1/100
 1/84 ━━━━━━━━━━━━━━━━━━━━ 4:49 3s/step - loss: 0.9924

W0000 00:00:1795602393.077971   53206 cpu_allocator_impl.cc:174] Allocation of 33030144 exceeds 10% of free system memory.
W0000 00:00:1795602393.077989   53204 cpu_allocator_impl.cc:174] Allocation of 33030144 exceeds 10% of free system memory.
W0000 00:00:1795602393.161362   53203 cpu_allocator_impl.cc:174] Allocation of 28835840 exceeds 10% of free system memory.


84/84 ━━━━━━━━━━━━━━━━━━━━ 18s 173ms/step - loss: 0.3054 - val_loss: 0.3320
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 13s 160ms/step - loss: 0.0847 - val_loss: 0.2620
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 163ms/step - loss: 0.0731 - val_loss: 0.2381
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 160ms/step - loss: 0.0685 - val_loss: 0.2320
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 21s 165ms/step - loss: 0.0658 - val_loss: 0.2322
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 163ms/step - loss: 0.0644 - val_loss: 0.2168
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 165ms/step - loss: 0.0627 - val_loss: 0.2137
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 166ms/step - loss: 0.0615 - val_loss: 0.2115
Epoch 9/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 20s 163ms/step - loss: 0.0607 - val_loss: 0.2095
Epoch 10/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 165ms/step - loss: 0.0600 - val_loss: 0.2073
Epoch 11/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14s 165ms/step - loss: 0.0594 - val_loss: 0.2087
Epoch 12/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 14

In [7]:
score = score_mse(modele_DAE, SBNorm)
seuil = float(np.percentile(score[i_train][sain[i_train]], 99))   # p99 des sains du train

modele_DAE.save(RACINE / "Sources" / "Fitted_models" / "DAE_fitted.keras")
joblib.dump({"seuil": seuil, "taille_fenetre": int(SB.shape[1])},   # 2048 avec duree=0.17067
            RACINE / "Sources" / "Fitted_models" / "DAE_params.joblib")
print(f"seuil p99 = {seuil:.4f}, fenêtre = {SB.shape[1]} points")

seuil p99 = 0.0096, fenêtre = 2048 points


In [8]:
# === Chargement du modèle entrainé === #
Fitted_Model = keras.models.load_model(RACINE / "Sources" / "Fitted_models" / "DAE_fitted.keras")

In [9]:
df_DAE = evaluer_par_capteur(Fitted_Model, "DAE (aug SNR 15..0)")
display(df_DAE)

,config,capteur,seuil_p99,AUC,FPR,FNR
0,DAE (aug SNR 15..0),DE,0.0096,1.0,0.0000,0.0
1,DAE (aug SNR 15..0),FE,0.0096,1.0,0.0211,0.0
